# Phase 8: Model Training — Ridge Regression & Naive Baseline

## 🎯 Objective
Establish initial multi-output forecasting benchmarks using the mandatory **Naive Persistence Baseline** ($\hat{y}_{t+h} = y_t$) and **Ridge Regression** (`MultiOutputRegressor(Ridge)`).

### Metric Hierarchy (Standardized):
- **Primary Decision Metric**: **Overall RMSE** (Root Mean Squared Error) across all 72 horizons.
- **Secondary Diagnostic Metrics**: **Overall MAE**, **Overall $R^2$**, and **Per-Horizon RMSE/MAE**.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.training_pipeline.trainer import ModelTrainer
from src.training_pipeline.evaluator import ModelEvaluator
from src.models.naive_baseline import NaivePersistenceBaseline
from src.models.ridge_model import RidgeAQIModel

MODELS_DIR = PROJECT_ROOT / "data" / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## 1. Train and Benchmark Models

In [ ]:
trainer = ModelTrainer(data_dir=PROCESSED_DIR, models_dir=MODELS_DIR)
ridge_model, df_comparison = trainer.run_ridge_pipeline()

print("\nTest Set Model Benchmark Summary:")
display(df_comparison)

## 2. Horizon-by-Horizon Error Degradation Curve ($h=1$ to $h=72$)

In [ ]:
with open(MODELS_DIR / "model_comparison.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

horizons = np.arange(1, 73)
naive_h_rmse = [eval_data["Naive Persistence Baseline"]["all_horizons"][f"h{h}"]["rmse"] for h in horizons]
ridge_h_rmse = [eval_data["Ridge Regression (MultiOutput)"]["all_horizons"][f"h{h}"]["rmse"] for h in horizons]

plt.figure(figsize=(11, 5))
plt.plot(horizons, naive_h_rmse, label="Naive Persistence Baseline", color="gray", linestyle="--", lw=2)
plt.plot(horizons, ridge_h_rmse, label="Ridge Regression (MultiOutput)", color="#4A90D9", lw=2.5)

plt.axvline(24, color="orange", linestyle=":", label="24h (Day 1)")
plt.axvline(48, color="green", linestyle=":", label="48h (Day 2)")
plt.axvline(72, color="red", linestyle=":", label="72h (Day 3)")

plt.title("Multi-Horizon Error Growth (RMSE vs Forecast Step $h$)")
plt.xlabel("Forecast Horizon (Hours ahead)")
plt.ylabel("Root Mean Squared Error (EPA AQI points)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Horizon Crossover Analysis

In [ ]:
rows = []
for h in horizons:
    k = f"h{h}"
    n_rmse = eval_data["Naive Persistence Baseline"]["all_horizons"][k]["rmse"]
    r_rmse = eval_data["Ridge Regression (MultiOutput)"]["all_horizons"][k]["rmse"]
    n_mae = eval_data["Naive Persistence Baseline"]["all_horizons"][k]["mae"]
    r_mae = eval_data["Ridge Regression (MultiOutput)"]["all_horizons"][k]["mae"]
    rows.append({
        "Horizon": h,
        "Naive RMSE": round(n_rmse, 2),
        "Ridge RMSE": round(r_rmse, 2),
        "Ridge Wins (RMSE)": r_rmse < n_rmse,
        "Naive MAE": round(n_mae, 2),
        "Ridge MAE": round(r_mae, 2),
    })

df_horizon = pd.DataFrame(rows)
wins = df_horizon["Ridge Wins (RMSE)"].sum()
print(f"Ridge beats Naive on RMSE in {wins}/72 horizons (Hours 1 through 37).")
display(df_horizon[df_horizon["Horizon"].isin([1, 6, 12, 24, 36, 48, 60, 72])])

## 4. In-Depth Metric Analysis & Horizon Degradation

### 4.1 Primary vs Secondary Metric Hierarchy
- **Primary Decision Metric**: **Overall RMSE** (82.97 for Ridge vs 84.05 for Naive Baseline, **+1.29% improvement**).
- **Secondary Diagnostic Metrics**: Overall MAE (63.14 vs 46.27), Overall $R^2$ (0.3740 vs 0.3576).

### 4.2 Why Ridge and Persistence Diverge on RMSE vs MAE
1. **$L_2$ Loss Minimization**: Ridge minimizes squared errors, aggressively curbing massive outlier errors during sharp pollution transitions. This drives down Overall RMSE and yields massive short-term error reductions (e.g. at $h+1$, Ridge achieves **53.18 RMSE vs 67.30**, a **21.0% error reduction**).
2. **Persistence Properties**: Persistence holds $\hat{y}_{t+h} = y_t$. When pollution changes slowly, median absolute deviations are small (low MAE), but when weather fronts or smog events shift pollution rapidly, persistence suffers massive squared errors.
3. **Horizon Crossover**: Ridge outperforms Persistence on RMSE for **horizons $h=1$ through $h=37$**. Beyond hour 37, linear Ridge coefficients shrink toward the historical training mean ($\approx 239$), causing error degradation at distant horizons ($h+72$ RMSE = 96.57 vs 88.31).

### 4.3 Conclusion for Phase 9 (Random Forest)
Linear regression cannot model non-linear smog thresholds and diurnal cycles at longer horizons. **Phase 9: Random Forest** will employ non-linear decision tree ensembles to improve both MAE and long-horizon accuracy.